In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
from scipy.signal import find_peaks, peak_widths, savgol_filter
import pandas as pd
import h5py
from typing import Literal
import struct

plt.rcParams["figure.dpi"] = 2000

In [ ]:
def plot_with_peaks(
    data,
    n_bins,
    prominence=None,
    xlabel="",
    title="",
    xlims=None,
    savgol_window=None,
    savgol_strength=1.0,
    clip_counts=None,
    stacked=False,
    legend=None,
    bin_range=None,
    text_size=5,
):
    if not stacked:
        signals = [data]
    else:
        signals = data

    bottom = np.zeros(n_bins)

    for signal in signals:
        counts, bins = np.histogram(signal, bins=n_bins, density=False, range=bin_range)
        if savgol_window is not None:
            background = savgol_filter(counts, window_length=savgol_window, polyorder=3)
            counts = counts - background * savgol_strength

        if clip_counts is not None:
            counts[counts > clip_counts] = clip_counts

        bin_width = np.diff(bins).mean()
        bin_centers = (bins[:-1] + bins[1:]) / 2.0
        height_scale = counts.max()

        plt.bar(bin_centers, counts, width=bin_width, align="center", bottom=bottom)
        plt.xlim(xlims)
        bottom += counts
        bottom = np.round(bottom, 10)
        if prominence is not None:
            peaks, props = find_peaks(counts, prominence=prominence)
            half_widths = peak_widths(counts, peaks)[0] * bin_width
            for x, y, half_width in zip(bin_centers[peaks], counts[peaks], half_widths):
                plt.errorbar(
                    [x],
                    [y],
                    xerr=[half_width],
                    c="black",
                    fmt="",
                    capsize=2,
                    elinewidth=0.2,
                    capthick=0.2,
                )
                plt.annotate(
                    f"{x:.03f} ± {half_width:.03f} MeV",
                    (x, y + height_scale * (0.005 * text_size)),
                    ha="left",
                    fontsize=text_size,
                )

    plt.xlabel(xlabel)
    plt.ylabel("Count")
    # plt.ylim(ylims)
    plt.title(title)
    if legend is not None:
        plt.legend(legend)

    # plt.show()

In [ ]:
# thx chatgpt

def fwhm(E):
    return np.sqrt(
        (5e-4)**2 +   # electronics noise
        0.000004 * E  # statistical broadening
    )

# Convert FWHM -> sigma
def sigma(E):
    return fwhm(E) / 2.355

# Loaders

In [ ]:
keys = [
    "Edep",
    "sourceE",
    "sourcePDG",
    "zInSoil",
    "rhoInSoil",
    # "sourceE"
]

def decode(b):
    return b.decode("utf-8")

decode_v = np.vectorize(decode)


def load_hdf5(path):
    dfs = []
    with h5py.File(path, "r") as f:
        for key in keys:
            data = f["default_ntuples"]["Measurement"][key]["pages"][()]
            if isinstance(data[0], bytes):
                data = decode_v(data)

            dfs.append(pd.DataFrame(data, columns=[key]))

    return pd.concat(dfs, axis=1)


def load_csv(path):
    df = pd.read_csv(path, comment="#", header=None)
    df.columns = keys.copy()
    print(f"{df.shape[0]:,} interactions recorded")
    return df


def load_raw(path):
    rows = []
    with open(path, "rb") as file:
        while data := file.read(3 * 8):
            energy, p_z, depth = struct.unpack("ddd", data)
            rows.append([energy, p_z, depth])
    df = pd.DataFrame(rows, columns=["E", "p_z", "z"])
    return df

In [ ]:
df = load_csv("archive/detection-1w-from-100M-all.csv")
df2 = load_csv("archive/detection-1w-from-2B-neutrons-only.csv")

In [ ]:
# thx chatgpt

range = df2["Edep"].between(1.0, 10.0)
# Smear each pulse energy
smeared = np.random.normal(
    loc=df2[range]["Edep"],
    scale=sigma(df2[range]["Edep"])
)
plot_with_peaks(smeared[(smeared > 4.4) & (smeared < 4.46)], 32)

# Analysis

## Masking Utilities

In [ ]:
property_range = lambda property, lower, upper, mask=None: (
    df if mask is None else df[mask]
)[property].between(lower, upper)
energy_range = lambda lower, upper, mask=None: property_range("Edep", lower, upper, mask)
include_particles = lambda names, mask=None: (df if mask is None else df[mask])[
    "name"
].isin(names)
exclude_particles = lambda names, mask=None: ~(df if mask is None else df[mask])[
    "name"
].isin(names)

## Energy Spectrum

In [ ]:
print(f"4.438 MeV count 1: {df["Edep"].between(4.437, 4.439).sum()}")
print(f"4.438 MeV count 2: {df2["Edep"].between(4.437, 4.439).sum()}")

In [ ]:
def plot_energy_spectrum(
    df, e_low, e_high, n_bins=512, prominence=None, clip=None, text_size=5
):
    energies = df[df["Edep"].between(e_low, e_high)]["Edep"]
    plot_with_peaks(
        data=energies,
        n_bins=n_bins,
        prominence=prominence,
        xlabel="Energy Deposited (MeV)",
        title="Germanium Detector Pulses",
        clip_counts=clip,
        text_size=text_size,
    )

plot_energy_spectrum(df, 2, 10, n_bins=4096, prominence=100, text_size=3)
plot_energy_spectrum(df2, 2, 10, n_bins=4096, prominence=100, text_size=3)
plt.show()
plot_energy_spectrum(df, 4.4, 4.46, n_bins=256, prominence=100)
plot_energy_spectrum(df2, 4.4, 4.46, n_bins=256, prominence=100)
plt.show()
plot_energy_spectrum(df, 6.10, 6.14, n_bins=256, prominence=100)
plot_energy_spectrum(df2, 6.10, 6.14, n_bins=256, prominence=100)
plt.show()
plot_energy_spectrum(df, 7.0, 7.2, n_bins=256, prominence=100)
plot_energy_spectrum(df2, 7.0, 7.2, n_bins=256, prominence=100)
plt.show()
plot_energy_spectrum(df, 5.5, 6.3, n_bins=1024, prominence=100)
plot_energy_spectrum(df2, 5.5, 6.3, n_bins=1024, prominence=100)
plt.show()

In [ ]:
plt.figure(figsize=(5, 7))

plt.title("Detection of Particles at Detector,\nby Position of Source in Soil")
plt.axis("off")

plt.subplot(2,1,1)

plt.hexbin(
    df["rhoInSoil"] / 10,
    df["zInSoil"] / 10,
    # C=df["Edep"],
    # reduce_C_function=np.sum,
    linewidths=0.2,
    # vmax=0.5,
    # norm=LogNorm(),
)
plt.xlabel("Radius from Origin, $\\rho$ (cm)")
plt.ylabel("Depth, $z$ (cm)")
plt.colorbar(label="Count")
# plt.xlim(0,250)
# plt.ylim(-41,0)
# plt.show()

plt.subplot(2,1,2)

plt.hexbin(
    df2["rhoInSoil"] / 10,
    df2["zInSoil"] / 10,
    # C=df["Edep"],
    # reduce_C_function=np.sum,
    linewidths=0.2,
    # vmax=0.5,
    # norm=LogNorm(),
)
plt.xlabel("Radius from Origin, $\\rho$ (cm)")
plt.ylabel("Depth, $z$ (cm)")
plt.colorbar(label="Count")
# plt.axis("off")
# plt.xlim(0,250)
# plt.ylim(-41,0)

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(5, 9))
plt.title("Detection Artefacts")
plt.axis("off")

plt.subplot(2,1,1)

mask = df["sourceE"].between(0.0, 5.0)
plt.hexbin(
    df[mask]["sourceE"],
    df[mask]["Edep"],
    # C=df["Edep"],
    # reduce_C_function=np.sum,
    linewidths=0.05,
    # vmax=100,
    norm=LogNorm(),
)
plt.colorbar(label="Count")
plt.xlabel("Gamma Ray Energy at Production (MeV)")
plt.ylabel("Energy Deposited in Detector (MeV)")
plt.xlim(0.0, 5.0)
plt.ylim(0, 5.0)

plt.subplot(2,1,2)

mask = df2["sourceE"].between(0.0, 5.0)
plt.hexbin(
    df2[mask]["sourceE"],
    df2[mask]["Edep"],
    # C=df["Edep"],
    # reduce_C_function=np.sum,
    linewidths=0.05,
    # vmax=100,
    norm=LogNorm(),
)
plt.colorbar(label="Count")
plt.xlabel("Gamma Ray Energy at Production (MeV)")
plt.ylabel("Energy Deposited in Detector (MeV)")
plt.xlim(0.0, 5.0)
plt.ylim(0, 5.0)

plt.show()